# Phase 1 

In [53]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold, LeaveOneOut
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
import numpy as np
import time

In [54]:
def split_train_val_test(X, y, test_size=0.2, val_size=0.2, random_state=42):
    if test_size <= 0 or val_size <= 0 or test_size + val_size >= 1:
        raise ValueError("test_size et val_size doivent être supérieurs à 0 et leur somme inférieure à 1.")

    X_reste, X_test, y_reste, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    proportion_val = val_size / (1 - test_size)
    X_train, X_val, y_train, y_val = train_test_split(
        X_reste, y_reste, test_size=proportion_val, random_state=random_state, stratify=y_reste
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

## Test 1 

In [55]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(X, y)

print("Train :", len(X_train))
print("Validation :", len(X_val))
print("Test :", len(X_test))
print("Total :", len(X_train) + len(X_val) + len(X_test))

Train : 341
Validation : 114
Test : 114
Total : 569


## Test 2 

In [56]:
try:
    split_train_val_test(X, y, val_size=0)
except ValueError as erreur:
    print("Erreur détectée :", erreur)

Erreur détectée : test_size et val_size doivent être supérieurs à 0 et leur somme inférieure à 1.


## Test 3 

In [57]:
indices = np.concatenate([
    np.where(y == 0)[0][:5],
    np.where(y == 1)[0][:95]
])

X_train_95, X_val_95, X_test_95, y_train_95, y_val_95, y_test_95 = split_train_val_test(X[indices], y[indices])

for nom, valeurs in [("Train", y_train_95), ("Validation", y_val_95), ("Test", y_test_95)]:
    print(nom, ":", np.bincount(valeurs), "soit", np.bincount(valeurs) / len(valeurs) * 100, "%")

Train : [ 3 57] soit [ 5. 95.] %
Validation : [ 1 19] soit [ 5. 95.] %
Test : [ 1 19] soit [ 5. 95.] %


# Phase 2

In [58]:
def bootstrap_scores(modele, X, y, n_iterations=30, random_state=42):
    rng = np.random.default_rng(random_state)
    scores = []

    for _ in range(n_iterations):
        indices = rng.choice(len(X), size=len(X), replace=True)
        indices_oob = np.setdiff1d(np.arange(len(X)), indices)

        if len(indices_oob) == 0:
            continue

        modele.fit(X[indices], y[indices])
        scores.append(modele.score(X[indices_oob], y[indices_oob]))

    print(f"Score moyen : {np.mean(scores):.3f} (± {np.std(scores):.3f})")
    return scores

## Test 1

In [59]:
modele = LogisticRegression(max_iter=10000)
scores = bootstrap_scores(modele, X, y)

Score moyen : 0.952 (± 0.015)


## Test 2

In [60]:
score_unique = bootstrap_scores(modele, X, y, n_iterations=1)

Score moyen : 0.920 (± 0.000)


# Phase 3

In [61]:
def evaluer_en_cross_val(modele, X, y, k=5):
    scores = cross_val_score(modele, X, y, cv=k, scoring="accuracy")
    print("Scores :", scores)
    print(f"Moyenne : {scores.mean():.3f} | Écart-type : {scores.std():.3f}")
    return scores

## Test 1

In [62]:
scores_cross_val = evaluer_en_cross_val(modele, X, y)

Scores : [0.93859649 0.94736842 0.98245614 0.92982456 0.95575221]
Moyenne : 0.951 | Écart-type : 0.018


## Test 2

In [63]:
debut = time.time()
scores_leave_one_out = cross_val_score(modele, X[:100], y[:100], cv=LeaveOneOut())
print(f"Leave-one-out sur 100 lignes : {time.time() - debut:.2f} secondes")

X_95 = X[indices]
y_95 = y[indices]
scores_non_stratifies = cross_val_score(modele, X_95, y_95, cv=KFold(5, shuffle=True, random_state=42))
scores_stratifies = cross_val_score(modele, X_95, y_95, cv=StratifiedKFold(5, shuffle=True, random_state=42))

print("Sans stratification :", scores_non_stratifies)
print("Avec stratification :", scores_stratifies)

Leave-one-out sur 100 lignes : 9.07 secondes
Sans stratification : [0.95 0.95 0.95 1.   1.  ]
Avec stratification : [0.95 1.   1.   0.95 0.95]


# Phase 4

In [64]:
def rapport_metier(y_true, y_pred, cout_fn=10, cout_fp=1):
    vn, fp, fn, vp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cout_total = fn * cout_fn + fp * cout_fp

    print("Matrice :", [[vn, fp], [fn, vp]])
    print(f"Precision={precision:.2f} Recall={recall:.2f} F1={f1:.2f} Coût={cout_total}")
    return cout_total

## Test 1

In [65]:
y_fraude = np.array([0] * 95 + [1] * 5)

prediction_a = np.array([0] * 98 + [1] * 2)
prediction_b = np.array([0] * 90 + [1] * 10)

print("Modèle A - Accuracy :", accuracy_score(y_fraude, prediction_a))
cout_a = rapport_metier(y_fraude, prediction_a)

print("\nModèle B - Accuracy :", accuracy_score(y_fraude, prediction_b))
cout_b = rapport_metier(y_fraude, prediction_b)

Modèle A - Accuracy : 0.97
Matrice : [[np.int64(95), np.int64(0)], [np.int64(3), np.int64(2)]]
Precision=1.00 Recall=0.40 F1=0.57 Coût=30

Modèle B - Accuracy : 0.95
Matrice : [[np.int64(90), np.int64(5)], [np.int64(0), np.int64(5)]]
Precision=0.50 Recall=1.00 F1=0.67 Coût=5


## Test 2

In [66]:
prediction_toujours_zero = np.zeros(100, dtype=int)

print("Accuracy :", accuracy_score(y_fraude, prediction_toujours_zero))
cout_zero = rapport_metier(y_fraude, prediction_toujours_zero)

Accuracy : 0.95
Matrice : [[np.int64(95), np.int64(0)], [np.int64(5), np.int64(0)]]
Precision=0.00 Recall=0.00 F1=0.00 Coût=50
